In [1]:
"""
FlyWire 连接组探索

这个脚本展示如何加载和可视化 FlyWire 连接组数据。
"""

'\nFlyWire 连接组探索\n\n这个脚本展示如何加载和可视化 FlyWire 连接组数据。\n'

In [2]:
import sys
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # 使用非交互式后端
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict, Counter

In [3]:
# 设置中文字体支持
import locale
locale.setlocale(locale.LC_ALL, '')

✓ 字体设置完成 (系统: Darwin)
  当前字体: ['Arial Unicode MS', 'PingFang SC', 'Heiti SC', 'STHeiti', 'DejaVu Sans']


In [4]:
import matplotlib.font_manager as fm
import platform
system = platform.system()

In [ ]:
if system == 'Darwin':  # macOS
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Heiti SC', 'STHeiti', 'DejaVu Sans']
elif system == 'Windows':
    plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'SimSun', 'DejaVu Sans']
else:  # Linux
    plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei', 'Noto Sans CJK SC', 'DejaVu Sans']

In [ ]:
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['text.usetex'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
# 设置绘图样式
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    try:
        plt.style.use('seaborn-darkgrid')
    except:
        pass
sns.set_palette("husl")

In [5]:
# 设置输出目录
output_dir = Path('outputs/connectome')
output_dir.mkdir(parents=True, exist_ok=True)

In [6]:
print("✓ 库导入成功")
print(f"✓ 输出目录: {output_dir}")

✓ 库导入成功
✓ 输出目录: outputs/connectome


============================================================================
1. 加载 FlyWire 连接组数据
============================================================================

In [7]:
print("\n" + "="*60)
print("1. 加载 FlyWire 连接组数据")
print("="*60)


1. 加载 FlyWire 连接组数据


In [8]:
# 加载 JSON 数据
json_path = Path("../flyvis/connectome/flywire_v1.0.json")

In [9]:
with open(json_path, 'r') as f:
    flywire_data = json.load(f)

In [10]:
print(f"✓ 加载 FlyWire 连接组: {json_path}")
print(f"  文件大小: {json_path.stat().st_size / 1024:.1f} KB")
print(f"\n数据结构:")
print(f"  - 节点数: {len(flywire_data['nodes'])}")
print(f"  - 边数: {len(flywire_data['edges'])}")
print(f"  - 输入单元: {flywire_data['input_units']}")
print(f"  - 输出单元: {flywire_data['output_units']}")

✓ 加载 FlyWire 连接组: ../flyvis/connectome/flywire_v1.0.json
  文件大小: 542.2 KB

数据结构:
  - 节点数: 146
  - 边数: 2071
  - 输入单元: ['R1-6', 'R7', 'R8']
  - 输出单元: ['T4a', 'T4b', 'T4c', 'T4d', 'T5a', 'T5b', 'T5c', 'T5d']


============================================================================
2. 基本统计信息
============================================================================

In [11]:
print("\n" + "="*60)
print("2. 基本统计信息")
print("="*60)


2. 基本统计信息


In [12]:
# 提取节点信息
nodes = flywire_data['nodes']
edges = flywire_data['edges']

In [13]:
# 细胞类型列表
cell_types = [node['name'] for node in nodes]
print(f"细胞类型总数: {len(cell_types)}")
print(f"\n前 20 个细胞类型:")
print(cell_types[:20])

细胞类型总数: 146

前 20 个细胞类型:
['Am1', 'CT1', 'Dm11', 'Dm13', 'Dm14', 'Dm15', 'Dm16', 'Dm17', 'Dm19', 'Dm6', 'Dm8a', 'Dm8b', 'DmDRA2', 'L2', 'L4', 'LLPt', 'LPi01', 'LPi02', 'LPi03', 'LPi04']


In [14]:
# 分析边的统计信息
edge_stats = {
    'total_edges': len(edges),
    'excitatory': sum(1 for e in edges if e.get('alpha', 1) == 1),
    'inhibitory': sum(1 for e in edges if e.get('alpha', 1) == -1),
    'with_offsets': sum(1 for e in edges if len(e.get('offsets', [])) > 0),
}

In [15]:
print("\n边统计:")
print(f"  总边数: {edge_stats['total_edges']}")
print(f"  兴奋性连接: {edge_stats['excitatory']} ({edge_stats['excitatory']/edge_stats['total_edges']*100:.1f}%)")
print(f"  抑制性连接: {edge_stats['inhibitory']} ({edge_stats['inhibitory']/edge_stats['total_edges']*100:.1f}%)")
print(f"  有空间偏移: {edge_stats['with_offsets']} ({edge_stats['with_offsets']/edge_stats['total_edges']*100:.1f}%)")


边统计:
  总边数: 2071
  兴奋性连接: 1418 (68.5%)
  抑制性连接: 653 (31.5%)
  有空间偏移: 2071 (100.0%)


In [16]:
# 计算每个细胞类型的连接数
in_degree = defaultdict(int)
out_degree = defaultdict(int)
total_synapses = defaultdict(int)

In [17]:
for edge in edges:
    src = edge['src']
    tar = edge['tar']
    out_degree[src] += 1
    in_degree[tar] += 1
    
    # 计算突触总数
    if 'offsets' in edge and len(edge['offsets']) > 0:
        syn_count = sum(offset[1] for offset in edge['offsets'])
    else:
        syn_count = 0
    total_synapses[(src, tar)] = syn_count

In [18]:
print("\n连接度最高的细胞类型:")
print("\n输出连接最多 (Top 10):")
for cell_type, count in sorted(out_degree.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {cell_type}: {count} 个目标")


连接度最高的细胞类型:

输出连接最多 (Top 10):
  TmY31: 45 个目标
  Tm7: 44 个目标
  TmY10: 36 个目标
  Tm5c: 35 个目标
  Sm42: 35 个目标
  Mi10: 34 个目标
  Mi4: 34 个目标
  Li33: 33 个目标
  TmY11: 33 个目标
  Tm20: 32 个目标


In [19]:
print("\n输入连接最多 (Top 10):")
for cell_type, count in sorted(in_degree.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {cell_type}: {count} 个来源")


输入连接最多 (Top 10):
  TmY31: 38 个来源
  Tm7: 35 个来源
  Li33: 35 个来源
  Tm5b: 34 个来源
  Li10: 33 个来源
  Tm5f: 33 个来源
  Tm5d: 30 个来源
  Li06: 30 个来源
  LLPt: 29 个来源
  Tm5a: 28 个来源


============================================================================
3. 可视化连接组结构
============================================================================

In [20]:
print("\n" + "="*60)
print("3. 可视化连接组结构")
print("="*60)


3. 可视化连接组结构


In [21]:
# 可视化兴奋性 vs 抑制性连接
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

In [22]:
# 饼图：兴奋性 vs 抑制性
labels = ['兴奋性', '抑制性']
sizes = [edge_stats['excitatory'], edge_stats['inhibitory']]
colors = ['#ff6b6b', '#4ecdc4']
axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
axes[0].set_title('连接类型分布', fontsize=14, fontweight='bold')

Text(0.5, 1.0, '连接类型分布')

In [23]:
# 柱状图：有/无空间偏移
offset_labels = ['有空间偏移', '无空间偏移']
offset_sizes = [edge_stats['with_offsets'], edge_stats['total_edges'] - edge_stats['with_offsets']]
axes[1].bar(offset_labels, offset_sizes, color=['#95e1d3', '#f38181'])
axes[1].set_ylabel('连接数', fontsize=12)
axes[1].set_title('空间偏移信息', fontsize=14, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

In [24]:
plt.tight_layout()
plt.savefig(output_dir / 'connection_types.png', dpi=300, bbox_inches='tight')
plt.close()

In [25]:
print(f"✓ 连接类型分布图已生成")

✓ 连接类型分布图已生成


In [26]:
# 可视化连接度分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

In [27]:
# 输入连接度分布
in_degrees = list(in_degree.values())
axes[0].hist(in_degrees, bins=30, color='#4ecdc4', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('输入连接数', fontsize=12)
axes[0].set_ylabel('细胞类型数量', fontsize=12)
axes[0].set_title('输入连接度分布', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)

In [28]:
# 输出连接度分布
out_degrees = list(out_degree.values())
axes[1].hist(out_degrees, bins=30, color='#ff6b6b', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('输出连接数', fontsize=12)
axes[1].set_ylabel('细胞类型数量', fontsize=12)
axes[1].set_title('输出连接度分布', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)

In [29]:
plt.tight_layout()
plt.savefig(output_dir / 'degree_distribution.png', dpi=300, bbox_inches='tight')
plt.close()

In [30]:
print(f"✓ 连接度分布图已生成")

✓ 连接度分布图已生成


============================================================================
4. 分析关键神经元类型
============================================================================

In [31]:
print("\n" + "="*60)
print("4. 分析关键神经元类型")
print("="*60)


4. 分析关键神经元类型


In [32]:
# 分析输入和输出神经元
input_types = flywire_data['input_units']
output_types = flywire_data['output_units']

In [33]:
print("输入神经元（光感受器）:")
for cell_type in input_types:
    out_conn = out_degree.get(cell_type, 0)
    print(f"  {cell_type}: 输出到 {out_conn} 个目标")

输入神经元（光感受器）:
  R1-6: 输出到 5 个目标
  R7: 输出到 11 个目标
  R8: 输出到 9 个目标


In [34]:
print("\n输出神经元（运动检测）:")
for cell_type in output_types:
    in_conn = in_degree.get(cell_type, 0)
    print(f"  {cell_type}: 接收来自 {in_conn} 个来源")


输出神经元（运动检测）:
  T4a: 接收来自 10 个来源
  T4b: 接收来自 10 个来源
  T4c: 接收来自 14 个来源
  T4d: 接收来自 10 个来源
  T5a: 接收来自 12 个来源
  T5b: 接收来自 14 个来源
  T5c: 接收来自 11 个来源
  T5d: 接收来自 14 个来源


============================================================================
5. 总结
============================================================================

In [35]:
print("\n" + "="*60)
print("FlyWire 连接组探索总结")
print("="*60)
print(f"\n✓ 成功加载 FlyWire 连接组")
print(f"  - 细胞类型: {len(cell_types)} 种")
print(f"  - 连接: {len(edges)} 个")
print(f"  - 兴奋性: {edge_stats['excitatory']} ({edge_stats['excitatory']/edge_stats['total_edges']*100:.1f}%)")
print(f"  - 抑制性: {edge_stats['inhibitory']} ({edge_stats['inhibitory']/edge_stats['total_edges']*100:.1f}%)")
print(f"  - 有空间偏移: {edge_stats['with_offsets']} ({edge_stats['with_offsets']/edge_stats['total_edges']*100:.1f}%)")
print(f"\n✓ 关键神经元")
print(f"  - 输入（光感受器）: {', '.join(input_types)}")
print(f"  - 输出（运动检测）: {', '.join(output_types)}")
print(f"\n🎉 探索完成！")
print("="*60)


FlyWire 连接组探索总结

✓ 成功加载 FlyWire 连接组
  - 细胞类型: 146 种
  - 连接: 2071 个
  - 兴奋性: 1418 (68.5%)
  - 抑制性: 653 (31.5%)
  - 有空间偏移: 2071 (100.0%)

✓ 关键神经元
  - 输入（光感受器）: R1-6, R7, R8
  - 输出（运动检测）: T4a, T4b, T4c, T4d, T5a, T5b, T5c, T5d

🎉 探索完成！


In [36]:
print(f"\n生成的图表文件:")
print(f"  - {output_dir / 'connection_types.png'}")
print(f"  - {output_dir / 'degree_distribution.png'}")


生成的图表文件:
  - outputs/connectome/connection_types.png
  - outputs/connectome/degree_distribution.png
